<a href="https://colab.research.google.com/github/aurora1112-j/aurora1112-j.github.io/blob/main/trade_register_combination.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SIPRI

In [1]:
import pandas as pd
import numpy as np

# ==========================================
# 第一步：读取与清洗 (Load & Clean)
# ==========================================
# 跳过前 11 行 (skiprows=11)，把第 12 行作为列名
df = pd.read_csv('trade-register.csv', skiprows=11)

# 清除列名的首尾空格
df.columns = df.columns.str.strip()

# 筛选冷战时期 (1950-1991)
# 语法结构: df[ (条件1) & (条件2) ]
df_cold_war = df[ (df['Delivery year'] >= 1950) & (df['Delivery year'] <= 1991) ].copy()

# ==========================================
# 第二步：构建分子 (Numerator: Flows)
# ==========================================
# 目标：计算 j (Supplier) 在 t 年卖给 i (Recipient) 的总额
# grouped：合并原始数据里同一个国家同一年可能有好几笔不同的武器交易（飞机、坦克分开记录）
df_flows = df_cold_war.groupby(['Delivery year', 'Recipient', 'Supplier'])['TIV delivery values'].sum().reset_index()

# 改个名字方便识别
df_flows.rename(columns={'TIV delivery values': 'TIV_flow'}, inplace=True)

# ==========================================
# 第三步：构建分母 (Denominator: Totals)
# ==========================================
# 计算 i (Recipient) 在 t 年的总进口额
# 注意：这里 groupby 少了一个 'Supplier'，因为要算的是“该国买的所有东西”
df_totals = df_flows.groupby(['Delivery year', 'Recipient'])['TIV_flow'].sum().reset_index()

# 改名为 Total_Import
df_totals.rename(columns={'TIV_flow': 'Total_Import'}, inplace=True)

# ==========================================
# 第四步：合并与计算 (Merge & Calculate)
# ==========================================
# merge 合并表格
# on=['Delivery year', 'Recipient']: 对齐年份和买家
# left：以左边为准，把年度总账对准每一笔交易（用来相除）
df_final = pd.merge(df_flows, df_totals, on=['Delivery year', 'Recipient'], how='left')

# 计算依赖度 W = Flow / (Total + epsilon)
# epsilon (1e-6) 是为了防止分母为 0 报错（虽然在这个数据里不太可能）
epsilon = 1e-6
df_final['Dependency_Ratio'] = df_final['TIV_flow'] / (df_final['Total_Import'] + epsilon)

# ==========================================
# 第五步：整理与保存 (Sort & Save)
# ==========================================
# 按年份、买家排序，依赖度高的排前面
df_final = df_final.sort_values(by=['Delivery year', 'Recipient', 'Dependency_Ratio'], ascending=[True, True, False])

# 保存结果
df_final.to_csv('cold_war_arms_dependency.csv', index=False)

# 打印前几行看看结果
print("处理完成！前 5 行数据如下：")
print(df_final.head())

处理完成！前 5 行数据如下：
   Delivery year  Recipient        Supplier  TIV_flow  Total_Import  \
0           1950    Albania    Soviet Union     19.60         19.60   
2           1950  Argentina   United States     80.04         84.84   
1           1950  Argentina           Italy      4.80         84.84   
3           1950  Australia  United Kingdom    464.30        464.30   
4           1950    Belgium  United Kingdom    163.10        198.16   

   Dependency_Ratio  
0          1.000000  
2          0.943423  
1          0.056577  
3          1.000000  
4          0.823072  


# ATOP + SIPRI

In [2]:
import pandas as pd
import numpy as np

# ==========================================
# 1. 核心映射字典 (手动构建)
# ==========================================
# 这是一个针对冷战优化的映射表，涵盖了主要的武器出口国和接收国
# 如果你发现报错说有些国家变成了 NaN，可以在这里手动添加
cow_codes_map = {
    # --- North America ---
    "United States": 2, "Canada": 20, "Bahamas": 31, "Cuba": 40, "Haiti": 41,
    "Dominican Republic": 42, "Jamaica": 51, "Trinidad and Tobago": 52,
    "Barbados": 53, "Mexico": 70, "Belize": 80, "Guatemala": 90, "Honduras": 91,
    "El Salvador": 92, "Nicaragua": 93, "Costa Rica": 94, "Panama": 95,

    # --- South America ---
    "Colombia": 100, "Venezuela": 101, "Guyana": 110, "Suriname": 115,
    "Ecuador": 130, "Peru": 135, "Brazil": 140, "Bolivia": 145, "Paraguay": 150,
    "Chile": 155, "Argentina": 160, "Uruguay": 165,

    # --- Europe ---
    "United Kingdom": 200, "Ireland": 205, "Netherlands": 210, "Belgium": 211,
    "Luxembourg": 212, "France": 220, "Switzerland": 225, "Spain": 230,
    "Portugal": 235, "Germany": 260,  # Mapping 'Germany' to West Germany (FRG) for Cold War
    "East Germany (GDR)": 265, "Poland": 290, "Austria": 305, "Hungary": 310,
    "Czechoslovakia": 315, "Italy": 325, "Albania": 339, "Yugoslavia": 345,
    "Greece": 350, "Cyprus": 352, "Bulgaria": 355, "Romania": 360,
    "Soviet Union": 365, "Russia": 365, "Finland": 375, "Sweden": 380,
    "Norway": 385, "Denmark": 390, "Iceland": 395, "Malta": 338,
    "Turkiye": 640, "Turkey": 640, # Map both modern and old spellings

    # --- Africa ---
    "Cape Verde": 402, "Cabo Verde": 402, "Sao Tome and Principe": 403,
    "Guinea-Bissau": 404, "Equatorial Guinea": 411, "Gambia": 420, "Mali": 432,
    "Senegal": 433, "Benin": 434, "Mauritania": 435, "Niger": 436,
    "Cote d'Ivoire": 437, "Guinea": 438, "Burkina Faso": 439, "Liberia": 450,
    "Sierra Leone": 451, "Ghana": 452, "Togo": 461, "Cameroon": 471,
    "Nigeria": 475, "Gabon": 481, "Central African Republic": 482, "Chad": 483,
    "Congo": 484, "DR Congo": 490, "Zaire": 490, "Uganda": 500, "Kenya": 501,
    "Tanzania": 510, "Burundi": 516, "Rwanda": 517, "Somalia": 520,
    "Djibouti": 522, "Ethiopia": 530, "Angola": 540, "Mozambique": 541,
    "Zambia": 551, "Zimbabwe": 552, "Malawi": 553, "South Africa": 560,
    "Namibia": 565, "Lesotho": 570, "Botswana": 571, "eSwatini": 572, "Swaziland": 572,
    "Madagascar": 580, "Mauritius": 590, "Seychelles": 591, "Morocco": 600,
    "Algeria": 615, "Tunisia": 616, "Libya": 620, "Sudan": 625,

    # --- Middle East ---
    "Iran": 630, "Iraq": 645, "Egypt": 651, "Syria": 652, "Lebanon": 660,
    "Jordan": 663, "Israel": 666, "Saudi Arabia": 670,
    "Yemen": 678, "North Yemen": 678, "Yemen Arab Republic (North Yemen)": 678,
    "South Yemen": 680, "Kuwait": 690, "Bahrain": 692, "Qatar": 694,
    "UAE": 696, "United Arab Emirates": 696, "Oman": 698,

    # --- Asia ---
    "Afghanistan": 700, "China": 710, "Mongolia": 712, "Taiwan": 713,
    "North Korea": 731, "South Korea": 732, "Japan": 740, "India": 750,
    "Bhutan": 760, "Pakistan": 770, "Bangladesh": 771, "Sri Lanka": 780,
    "Nepal": 790, "Thailand": 800, "Cambodia": 811, "Laos": 812,
    "Viet Nam": 816, "North Vietnam": 816, "South Vietnam": 817,
    "Malaysia": 820, "Singapore": 830, "Brunei": 835, "Philippines": 840,
    "Indonesia": 850, "Papua New Guinea": 910, "Australia": 900,
    "New Zealand": 920, "Fiji": 950, "Solomon Islands": 940, "Vanuatu": 935,
    "Tonga": 955, "Myanmar": 775, "Burma": 775, "Comoros": 581,

    # --- Non-State Actors / Rebels (Map to None/NaN) ---
    # You cannot merge these with ATOP state data.
    "ANC (South Africa)*": None, "Anti-Castro rebels (Cuba)*": None,
    "Contras (Nicaragua)*": None, "FMLN (El Salvador)*": None,
    "Hezbollah (Lebanon)*": None, "Khmer Rouge (Cambodia)*": None,
    "Mujahideen (Afghanistan)*": None, "PLO (Israel)*": None,
    "Taliban (Afghanistan)*": None, "UNITA (Angola)*": None,
    "Viet Cong (South Vietnam)*": None, "NATO**": None, "United Nations**": None
}

# ==========================================
# 2. 读取数据
# ==========================================
# 请确保文件名与你本地一致
df_atop = pd.read_csv('atop_clean.csv')
df_arms = pd.read_csv('cold_war_arms_dependency.csv')

print("正在读取数据...")

# ==========================================
# 3. 转换国家名为 COW 编码 (使用 .map)
# ==========================================
print("正在转换国家名称...")

# 将 Supplier (出口国) 映射为 stateA
df_arms['stateA'] = df_arms['Supplier'].map(cow_codes_map)

# 将 Recipient (进口国) 映射为 stateB
df_arms['stateB'] = df_arms['Recipient'].map(cow_codes_map)

# --- 关键调试步骤 ---
# 检查有哪些国家没有被成功转换 (即 map 结果为 NaN)
missing_suppliers = df_arms[df_arms['stateA'].isna()]['Supplier'].unique()
missing_recipients = df_arms[df_arms['stateB'].isna()]['Recipient'].unique()

if len(missing_suppliers) > 0 or len(missing_recipients) > 0:
    print("\n⚠️ 注意：以下国家名称未在字典中找到，将被丢弃：")
    if len(missing_suppliers) > 0:
        print(f"缺失的出口国 (前5个): {missing_suppliers[:5]}")
    if len(missing_recipients) > 0:
        print(f"缺失的接收国 (前5个): {missing_recipients[:5]}")
else:
    print("完美！所有国家都成功匹配。")

# 剔除无法识别的国家
df_arms = df_arms.dropna(subset=['stateA', 'stateB'])
df_arms['stateA'] = df_arms['stateA'].astype(int)
df_arms['stateB'] = df_arms['stateB'].astype(int)

# 统一列名
df_arms = df_arms.rename(columns={'Delivery year': 'year', 'Dependency_Ratio': 'weight'})

# ==========================================
# 4. 执行合并 (Layer 1 完成体)
# ==========================================
print("\n正在合并条约与武器数据...")

# Outer Join: 保留任一来源的连接
merged_df = pd.merge(
    df_atop[['stateA', 'stateB', 'year', 'weight']],
    df_arms[['stateA', 'stateB', 'year', 'weight']],
    on=['stateA', 'stateB', 'year'],
    how='outer',
    suffixes=('_atop', '_arms')
)

# 填充缺失值
merged_df['weight_atop'] = merged_df['weight_atop'].fillna(0)
merged_df['weight_arms'] = merged_df['weight_arms'].fillna(0)

# 计算最终权重: 取最大值
merged_df['final_security_weight'] = merged_df[['weight_atop', 'weight_arms']].max(axis=1)

# 排序
merged_df = merged_df.sort_values(by=['year', 'stateA', 'stateB'])

# ==========================================
# 5. 保存
# ==========================================
output_file = 'name_to_cow.csv'
merged_df.to_csv(output_file, index=False)

print("-" * 30)
print(f"合并成功！文件已保存至: {output_file}")
print(f"总数据行数: {len(merged_df)}")


正在读取数据...
正在转换国家名称...

⚠️ 注意：以下国家名称未在字典中找到，将被丢弃：
缺失的出口国 (前5个): ['unknown supplier(s)' 'United Nations**' 'FMLN (El Salvador)*']
缺失的接收国 (前5个): ['Armas (Guatemala)*' 'Viet Minh (France)*' 'NATO**' 'Indonesia rebels*'
 'unknown recipient(s)']

正在合并条约与武器数据...
------------------------------
合并成功！文件已保存至: name_to_cow.csv
总数据行数: 82577
